# Synthetic Auxiliary T2 Adapter

This workflow trains a separate text-only T2 adapter on synthetic auxiliary records. It must not be presented as clinical validation or veterinary advice.

In [ ]:
%pip install -q -U "transformers>=4.50.0" peft accelerate bitsandbytes pillow scikit-learn safetensors shap
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or T4 GPU runtime before continuing")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import getpass
import os
import tarfile

uploaded = files.upload()
bundle_name = next(name for name in uploaded if name.endswith(".tar.gz"))
root = Path("/content/t2_bundle")
root.mkdir(exist_ok=True)
with tarfile.open(Path("/content") / bundle_name, "r:gz") as archive:
    archive.extractall(root)
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste HF_TOKEN: ")
adapter = Path("/content/t2_adapter")
results = Path("/content/t2_results")
results.mkdir(exist_ok=True)
print(root)

In [ ]:
import subprocess
import sys

runner = root / "scripts"
subprocess.run([
    sys.executable, "-u", str(runner / "train_t2_qlora.py"),
    "--root", str(root),
    "--output", str(adapter),
    "--epochs", "1",
    "--learning-rate", "2e-4",
    "--gradient-accumulation-steps", "8",
], check=True)

In [ ]:
subprocess.run([
    sys.executable, "-u", str(runner / "run_t2_generation.py"),
    "--root", str(root),
    "--input", str(root / "data/secondary/instruction_tuning_synthetic/val.jsonl"),
    "--adapter", str(adapter),
    "--output", str(results / "t2_val_generation.jsonl"),
    "--max-examples", "151",
    "--max-new-tokens", "128",
], check=True)
subprocess.run([
    sys.executable, "-u", str(runner / "run_t2_generation.py"),
    "--root", str(root),
    "--input", str(root / "data/secondary/instruction_tuning_synthetic/hallucination_probe.jsonl"),
    "--adapter", str(adapter),
    "--output", str(results / "t2_probe_generation.jsonl"),
    "--max-examples", "60",
    "--max-new-tokens", "128",
], check=True)

In [ ]:
subprocess.run([
    sys.executable, str(runner / "evaluate_t2_generation.py"),
    "--input", str(results / "t2_val_generation.jsonl"),
    "--out", str(results / "t2_val_metrics.json"),
], check=True)
subprocess.run([
    sys.executable, str(runner / "evaluate_t2_generation.py"),
    "--input", str(results / "t2_probe_generation.jsonl"),
    "--out", str(results / "t2_probe_metrics.json"),
], check=True)

In [ ]:
subprocess.run([
    sys.executable, "-u", str(runner / "attribute_t2_occlusion.py"),
    "--input", str(root / "data/secondary/instruction_tuning_synthetic/val.jsonl"),
    "--adapter", str(adapter),
    "--output", str(results / "t2_shap.jsonl"),
    "--max-examples", "4",
    "--method", "shap",
    "--shap-samples", "32",
], check=True)
import shutil
archive = shutil.make_archive("/content/t2-adapter-results", "zip", root_dir=results.parent, base_dir=results.name)
print(archive)
files.download(archive)